# 03 - Feature engineering (part 1)

Adds the features that need no text parsing, then gives an overview of the resulting data.

| Group | Features | Built from |
|---|---|---|
| Time (UTC) | `utc_date`, `utc_month`, `utc_iso_week`, `utc_weekday`, `utc_weekday_name`, `utc_is_weekend`, `utc_hour` | `epoch_seconds` |
| Parties | `party_pair` | `ordering_type`, `beneficiary_type` |
| Amount | `usd_amount_log10`, `usd_amount_magnitude`, `usd_amount_is_whole` | `usd_amount` |

**Input:** `data/processed/transactions_clean.parquet` (run `01_preprocessing.ipynb` first).  
**Output:** `data/processed/transactions_features.parquet` = every clean column, untouched, plus the new columns.

Geography, routing, account, token and anomaly-signature features are added later, once the text-column formats are confirmed in `02` (EDA).

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

# Works whether Jupyter was started in the project root or in notebooks/
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
assert (ROOT / "src").exists(), f"Project root not found from {Path.cwd()}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import features
from src import settings as S

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 100)
print("Project root:", ROOT)

## 1. Build the features

Processed in batches, so memory use stays low. The original columns are passed through unchanged.

In [ ]:
features.run()

What each new column means:

In [ ]:
pd.read_csv(features.FEATURE_DICTIONARY_CSV)

**Notes on interpretation**

- All time features are in **UTC**, not the local time of the sender or receiver. A payment at 09:00 in Kuala Lumpur shows as `utc_hour = 1`. Local-time features come later, with the geography group.
- `utc_is_weekend` means Saturday or Sunday in UTC. Around midnight UTC this can differ from the local day, and some countries have a different weekend.
- `usd_amount` is already converted to USD, so `usd_amount_is_whole` only shows a round amount reliably when the payment currency is USD. For other currencies a round original amount usually stops being round after conversion.

## 2. Overview of the data after the process

### 2.1 Size

In [ ]:
pf = pq.ParquetFile(features.FEATURES_PARQUET)
schema = pf.schema_arrow
new_cols = [f[0] for f in features.FEATURES]
original_cols = [c for c in schema.names if c not in new_cols]

print(f"Rows            : {pf.metadata.num_rows:,}")
print(f"Columns         : {len(schema.names)}  ({len(original_cols)} from the clean file + {len(new_cols)} new)")
print(f"Clean file      : {S.CLEAN_PARQUET.stat().st_size / 1e6:,.1f} MB")
print(f"Features file   : {features.FEATURES_PARQUET.stat().st_size / 1e6:,.1f} MB")

### 2.2 All columns

In [ ]:
pd.DataFrame({
    "column": schema.names,
    "type": [str(t) for t in schema.types],
    "origin": ["new feature" if c in new_cols else "clean file" for c in schema.names],
})

### 2.3 First 10 rows

In [ ]:
next(pf.iter_batches(batch_size=10)).to_pandas()

### 2.4 New features: missing values and range

Only the new feature columns (plus the label) are loaded here, so this stays light on memory.

In [ ]:
extra = [c for c in ["anomaly_type", "is_anomaly"] if c in schema.names]
df = pq.read_table(features.FEATURES_PARQUET, columns=new_cols + extra).to_pandas(date_as_object=False)
print(f"{len(df):,} rows x {df.shape[1]} columns, {df.memory_usage(deep=True).sum() / 1e6:,.1f} MB in memory")

rows = []
for c in new_cols:
    s = df[c]
    ordered = s.dtype.kind in "iufM"  # numbers and dates have a meaningful min/max
    rows.append({
        "feature": c,
        "dtype": str(s.dtype),
        "missing": int(s.isna().sum()),
        "distinct": int(s.nunique()),
        "min": s.min() if ordered else "",
        "max": s.max() if ordered else "",
    })
pd.DataFrame(rows)

### 2.5 Time coverage

In [ ]:
per_day = df.groupby("utc_date").size()
print(f"First day   : {per_day.index.min().date()}")
print(f"Last day    : {per_day.index.max().date()}")
print(f"Days covered: {len(per_day)} (calendar span {(per_day.index.max() - per_day.index.min()).days + 1} days)")
print(f"Payments per day: min {per_day.min():,} | median {int(per_day.median()):,} | max {per_day.max():,}")
print("Note: the first and last day can be partial.")

### 2.6 Distribution of each new feature

In [ ]:
def counts(col, sort_index=True):
    t = df[col].value_counts(dropna=False)
    t = t.sort_index() if sort_index else t
    return pd.DataFrame({"rows": t, "pct": (t / len(df) * 100).round(4)})

display(counts("party_pair"))
display(counts("utc_month"))
display(counts("utc_weekday_name"))
display(counts("utc_is_weekend"))

In [ ]:
display(counts("utc_hour"))
display(counts("utc_iso_week"))

In [ ]:
display(counts("usd_amount_magnitude"))
display(counts("usd_amount_is_whole"))
df["usd_amount_log10"].describe().to_frame().T

### 2.7 Quick check against the brief

The brief defines a romance scam as consumer to consumer, and a CEO scam as a payment made by a business. If `party_pair` is built correctly, the labelled rows should show that pattern.

In [ ]:
if "anomaly_type" in df.columns:
    display(pd.crosstab(df["anomaly_type"], df["party_pair"], margins=True, margins_name="Total"))
else:
    print("anomaly_type is not in the data - skipped")